# Problem 988 - Non-Attacking Frogs
Frogs can be placed on the real number line at integer locations. Given coprime positive integers $(a,b)$, each frog has the ability to make jumps of distances $a$ or $b$ in the positive direction.

Two frogs placed at $m$ and $n$, $m<n$, are <i>attacking</i> if the frog at $m$ can hop to $n$ with some series of jumps. For example if $(a,b)=(3,5)$, frogs placed at $0$ and $11$ are attacking as the former can make two jumps of $3$ and one jump of $5$ to reach $11$. However, frogs placed at $4$ and $11$ are non-attacking.

A <i>non-attacking configuration</i> is a placement of any number of frogs such that:

- one frog is placed at $0$;
- all other frogs are placed at distinct positive integers;
- no two frogs are attacking.

Define $F(a,b)$ to be sum of the integer locations of every frog, summing over all non-attacking configurations. For example if $(a,b)=(3,5)$ there are seven non-attacking configurations:
$$\{0\}\quad\quad\{0,1\}\quad\quad\{0,2\}\quad\quad\{0,4\}\quad\quad\{0,7\}\quad\quad\{0,1,2\}\quad\quad\{0,2,4\}
$$giving $F(3,5)=23$.


You are also given $F(5,13)=16336$.

Find $F(19,53)$.

## Solution.

Let us use: https://en.wikipedia.org/wiki/Coin_problem

In [13]:
def attackable(a, b):
    limit = (a-1)*(b-1)
    ans = set()

    for x in range(limit//a + 1):
        max_y = (limit - a*x - 1)//b
        for y in range(max_y + 1):
            ans.add(a*x + b*y)

    return ans - set([0])

In [14]:
def F(a, b):
    limit = (a-1)*(b-1)

    A = attackable(a, b) # places that attackable from 0
    B = set([x for x in range(1, limit, 1)]) - A # places that not attackable from 0, thus potential elements of the set

    attacking = dict()

    for b in B:
        D = set()

        for a in A:
            if a + b in B: # if a+b is in B, then b can attack this member of B
                D.add(a+b)
        attacking[b] = D
        
    return attacking

In [16]:
F(19, 53)

{1: {20,
  39,
  54,
  58,
  73,
  77,
  92,
  96,
  107,
  111,
  115,
  126,
  130,
  134,
  145,
  149,
  153,
  160,
  164,
  168,
  172,
  179,
  183,
  187,
  191,
  198,
  202,
  206,
  210,
  213,
  217,
  221,
  225,
  229,
  232,
  236,
  240,
  244,
  248,
  251,
  255,
  259,
  263,
  267,
  270,
  274,
  278,
  282,
  286,
  289,
  293,
  297,
  301,
  305,
  308,
  312,
  316,
  320,
  324,
  327,
  331,
  335,
  339,
  343,
  346,
  350,
  354,
  358,
  362,
  365,
  369,
  373,
  377,
  381,
  384,
  388,
  392,
  396,
  400,
  403,
  407,
  411,
  415,
  419,
  422,
  426,
  430,
  434,
  438,
  441,
  445,
  449,
  453,
  457,
  460,
  464,
  468,
  472,
  476,
  479,
  483,
  487,
  491,
  495,
  498,
  502,
  506,
  510,
  514,
  517,
  521,
  525,
  529,
  533,
  536,
  540,
  544,
  548,
  552,
  555,
  559,
  563,
  567,
  571,
  574,
  578,
  582,
  586,
  590,
  593,
  597,
  601,
  605,
  609,
  612,
  616,
  620,
  624,
  628,
  631,
  635,
  639,
  643,
  64

In [20]:
def edge_free_subgraph_vertex_sum(graph: dict) -> int:
    # build undirected adjacency
    adj = {v: set() for v in graph}
    for v in graph:
        for u in graph[v]:
            adj.setdefault(u, set()).add(v)
            adj[v].add(u)

    # find connected components via BFS
    def get_components():
        visited = set()
        components = []
        for start in adj:
            if start in visited:
                continue
            comp = []
            queue = [start]
            visited.add(start)
            while queue:
                v = queue.pop()
                comp.append(v)
                for u in adj[v]:
                    if u not in visited:
                        visited.add(u)
                        queue.append(u)
            components.append(comp)
        return components

    return get_components()

    def solve_component(vertices):
        n = len(vertices)
        local_adj = {v: adj[v] & set(vertices) for v in vertices}

        # sort by degree descending for better pruning
        vertices = sorted(vertices, key=lambda v: len(local_adj[v]), reverse=True)
        vidx = {v: i for i, v in enumerate(vertices)}
        excluded = [False] * n

        # returns (sum_of_all_IS, count_of_all_IS) including empty set
        def backtrack(idx):
            if idx == n:
                return (0, 1)

            v = vertices[idx]

            if excluded[idx]:
                return backtrack(idx + 1)

            # exclude v
            ex_sum, ex_count = backtrack(idx + 1)

            # include v
            changed = []
            for u in local_adj[v]:
                i = vidx[u]
                if not excluded[i]:
                    excluded[i] = True
                    changed.append(i)
            in_sum, in_count = backtrack(idx + 1)
            in_sum += v * in_count
            for i in changed:
                excluded[i] = False

            return (ex_sum + in_sum, ex_count + in_count)

        return backtrack(0)

    # combine results across components
    # (s_a, c_a) x (s_b, c_b) -> (s_a*c_b + s_b*c_a, c_a*c_b)
    total_sum, total_count = 1, 1  # neutral element: empty set only
    total_sum, total_count = 0, 1

    for comp in get_components():
        c_sum, c_count = solve_component(comp)
        total_sum  = total_sum * c_count + c_sum * total_count
        total_count = total_count * c_count

    # subtract empty set (contributes 0 to sum, so only count matters — no correction needed)
    return total_sum


graph = {1: {2}, 2: {3, 4}, 3: set(), 4: set()}
print(edge_free_subgraph_vertex_sum(graph))  # 34

[[1, 2, 4, 3]]


In [21]:
edge_free_subgraph_vertex_sum(F(5, 13))

[[1,
  29,
  9,
  22,
  17,
  12,
  7,
  2,
  4,
  3,
  8,
  27,
  24,
  21,
  19,
  16,
  47,
  14,
  11,
  42,
  6,
  37,
  34,
  32]]

In [22]:
edge_free_subgraph_vertex_sum(F(19,53))

[[1,
  510,
  404,
  499,
  446,
  503,
  450,
  507,
  454,
  511,
  458,
  439,
  420,
  405,
  386,
  367,
  352,
  333,
  314,
  299,
  280,
  261,
  246,
  227,
  208,
  193,
  174,
  155,
  140,
  121,
  102,
  87,
  68,
  49,
  34,
  15,
  492,
  473,
  435,
  416,
  401,
  382,
  363,
  348,
  329,
  310,
  295,
  276,
  257,
  242,
  223,
  204,
  189,
  170,
  151,
  136,
  117,
  98,
  83,
  64,
  45,
  30,
  11,
  488,
  469,
  564,
  545,
  526,
  431,
  412,
  397,
  378,
  359,
  344,
  325,
  306,
  291,
  272,
  253,
  238,
  219,
  200,
  185,
  166,
  147,
  132,
  113,
  94,
  79,
  60,
  41,
  26,
  7,
  484,
  465,
  617,
  598,
  579,
  560,
  541,
  522,
  427,
  408,
  393,
  389,
  374,
  370,
  355,
  340,
  336,
  321,
  317,
  302,
  287,
  283,
  268,
  264,
  249,
  234,
  230,
  215,
  211,
  196,
  181,
  177,
  162,
  158,
  143,
  128,
  124,
  109,
  105,
  90,
  75,
  71,
  56,
  52,
  37,
  22,
  18,
  3,
  480,
  461,
  442,
  423,
  670,
  651,
 